# Q1. Object detection and depth estimation                             20+5+20 = 45 points
 Use L515 camera in Lab to acquire rgb video and depth video (e.g. 30 s) of swinging load in lab (rgb_train_val_swingingload.avi; depth_train_val_swingingload.avi). Extract each frames from rgb video. Use the extracted images to train and validate YOLO model (save as YOLO_swinging_load.pt). Record test_swingingload.avi (e.g. 5 s). Extract frames and use them as test data set. Test your YOLO model (YOLO_swinging_load.pt) on this test data set.

(a) Show the performance of your model on test data set by plotting relevant KPIs.

(b) Sample five random images from test data video (test_swingingload.avi). Find the center, and coordinates of bounding box. Display them on the each test image.

(c) realsense_acq.ipynb contains code that streams, rgb, and depth video. Based on this code, extract the frames from depth_train_val_swingingload.avi. Each pixel in each frame consists of depth information of corresponding rgb images. As described in Q1.b find the center of bounding box of ALL frames in  test video (test_swingingload.avi). Display its center and corresponding depth.


References: Use Ultralytics framework for help - https://docs.ultralytics.com/tasks/detect
Realsense - https://dev.realsenseai.com/sdk-2-0-code-samples-wrappers-and-languages/opencv/ 

Data acquisition: realsense_acq.ipynb contains the code to acquire RGB and Depth data from L515.
 
Cell 1: Streams RGB and Depth data

Cell 2: Streams RGB and Depth data, and saves the following:
- session_date_time e.g. session_20260914_1354
  - /color_video. Contains colour RGB video
  - /depth_video. Contains depth video video
  - /depth_raw. Contains raw depth value
  - /depth_scale_json. Contains scaling factor. multiply this scaling factor with raw scale to obtain true depth in meter
  -/rgb_frames. Contains RGB frames that you can use for labeling and object detection
  

In [ ]:
# Q1 Prep:
# Pull in ultralytics pretrained model, train on custom dataset, save model

# Dataset preparation required:
# - Moving all rgb_frames into separate folders in q1/dataset/ 
# - Split training images 90% to keep and move 10% into val/ sequentially to avoid data leakage
# - Label all .png using Labelme to generate Labelme .JSON labels (Took me 6 hours!)
# - Move all labels into labels/ directory
# - Convert all JSON labels to YOLO formatted .txt using conversion script from Gemini
# 
# dataset/
#    |- images/
#        |- train/ -> rgb_0000.png - rgb_0809.png
#        |- test/ -> rgb_0000.png - rgb_0299.png
#        |- val/ -> rgb_0810.png - rgb_0899.png
#    |- labels/
#        |- train/ ->  rgb_0000.txt - rgb_0809.txt
#        |- test/ -> rgb_0000.txt - rgb_0299.txt
#        |- val/ -> rgb_0810.txt - rgb_0899.txt
#    |- dataset.yaml
#    |- labelme2yolo.py

from ultralytics import YOLO

# Want to test yolo26n because this is a model we will be using in SeaBotics, attempting transfer of knowledge from curriculum to real-life
# Higher "tier" model such as yolo26x would have higher accuracy, but takes longer to train and is not small enough for edge computing-
# as desired in SeaBotics project (Luxonis OAK-D Pro AI Camera).

model = YOLO("yolo26n.pt")

results = model.train(
    data="q1/dataset/dataset.yaml",
    epochs=100,
    imgsz=(640, 480), # -- Images are 640x480, not 640x640
    ) 

# Have edited out test/ from dataset.yaml for training to avoid accidentally exposing it to model during training
# Now run model and it will automatically create a best.pt which can be copied and renamed to YOLO_swinging_load.pt in q1/model

In [ ]:
# Q1 A
# Uncomment test directory from dataset.yaml before running

from ultralytics import YOLO
import cv2
import os

model = YOLO("./q1/model/YOLO_swinging_load.pt")

metrics = model.val(data="./q1/dataset/dataset.yaml", split="test")

# Plot metrics
print(metrics.box.results_dict) 

# Visualize predictions on test video
video = "./q1/data/test/color_video/color_video.avi"
cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

os.makedirs("q1/inferred", exist_ok=True)
video_writer = cv2.VideoWriter("q1/inferred/YOLO_swinging_load_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame)
    video_writer.write(results[0].plot())

cap.release()
video_writer.release()
cv2.destroyAllWindows()



In [ ]:
# Q1 B 

from ultralytics import YOLO
import os
import numpy as np

images = []
output_images = []
np.random.seed(69) # Nice
dir1 = "./q1/dataset/images/test"
dir2 = "./q1/inferred"

os.makedirs(dir2, exist_ok=True)

for i in range(5):
    # Generate a random number between 0 and 299
    # Fetch an image at random number position from given image test directory
    # Add that image to images array 

    img_number = np.random.randint(0, 300) # -- amount of image files in test data set, 000 - 299
    image_name = f"rgb_{img_number:04d}.png"
    images.append(os.path.join(dir1, image_name))

# Load YOLO_swinging_load.pt 
model = YOLO("q1/model/YOLO_swinging_load.pt")
counter = 0

for image in images:
    # Run actual inference using model on each image
    results = model(image)

    # Store inferred image and bounding box in output_images and on new .png files
    output_images.append(results[0])
    results[0].save(os.path.join(dir2,f"{counter}.png"))
    counter += 1
    
for output in output_images:
    # Display inferred images with bounding boxes
    output.show()

# Q2. Object tracking                                                           5 points
  Perform tracking of swinging load on test_swingingload.avi. You should display the trajectory of load as it swings each frame as shown in Fig. below. 

![image](q2\Q2_example.png)

Ref: Use help from Ultralytics - https://docs.ultralytics.com/guides/instance-segmentation-and-tracking 

In [ ]:
# Q2
# https://docs.ultralytics.com/guides/instance-segmentation-and-tracking
from ultralytics import solutions
import cv2
import matplotlib.pyplot as plt

model = YOLO(".q1/model/YOLO_swinging_load.pt") 
cap = cv2.VideoCapture("./q1/data/test/color_video/color_video.avi") # Assuming this is meant to be test_swinging_load_video.avi

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

# Create plot 
fig = plt.figure(figsize=(750, 455))

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = model(im0)
    plt.plot(results[0].boxes.xywh[:, 0], results[0].boxes.xywh[:, 1], 'ro')  # Plot center coordinates
    
cv2.imwrite("q2/output/instance_segmentation_position_output.png", fig)
cap.release()
cv2.destroyAllWindows() 


Ultralytics Solutions:  {'source': None, 'model': 'yolo26n-seg.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 640x640 19.9ms, 1 fire hydrant
Speed: 435.4ms track, 19.9ms solution per image at shape (1, 3, 640, 640)



AttributeError: 'SolutionResults' object has no attribute 'get_enclosing_boxes'

Error in callback <function flush_figures at 0x000001C4AF780220> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

# Q3. Multi object segmentation, tracking and counting:                         10  points

Explore different data-set available freely on internet (https://docs.ultralytics.com/datasets ). Get familiar with them. Then choose a data set and corresponding class of your choice (e.g. CIFAR-10, dataset/COCO data set or KITTI data set / person). Then make a video such that multiple instances of objects are in the scene (e.g. 5 people are the scene). Perform 
(a) instance segmentation, 
(b) counting  - total number of object in scene
(c) and tracking of individual instance of object
(d) draw a region - and count the object in that region

In [ ]:
# Q3 A
# https://docs.ultralytics.com/tasks/segment

from ultralytics import YOLO
import cv2

# Load a COCO pretrained YOLO model for instance segmentation
model = YOLO("yolo26n-seg.pt")

video = "./q3/video/people.mp4"
cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("q3/results/object_segmentation_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame)
    video_writer.write(results[0].plot())

cap.release()
video_writer.release()
cv2.destroyAllWindows()

In [ ]:
# Q3 B
# https://docs.ultralytics.com/guides/object-counting

import cv2
from ultralytics import solutions, YOLO

video = "./q3/video/people.mp4"
cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

region_points = [(0, 0), (w, 0), (w, h), (0, h)]

counter = solutions.ObjectCounter( 
    region=region_points, 
    model="yolo26n.pt",
    show_out=False,
    classes=[0],
)

frame_counter = 0
frame_nmbr = 2

# Only want to count objects in scene once, i.e one frame of video
while frame_counter < frame_nmbr:
    success, im0 = cap.read()

    if not success:
        break

    results = counter(im0)

    if frame_counter == frame_nmbr - 1:    
        # Create png of frame with bounding boxes and save to q3/results
        image = results.plot_im
        cv2.imwrite("q3/results/object_counting_output.png", image)
        frame_counter += 1
    else:
        frame_counter += 1
        continue
        

cap.release()
cv2.destroyAllWindows()



In [ ]:
# Q3 C
# https://docs.ultralytics.com/guides/instance-segmentation-and-tracking

import cv2
from ultralytics import solutions

video = "./q3/video/people.mp4"
cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("q3/results/object_segmentation_tracking_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))


isegment = solutions.InstanceSegmentation(
    model="yolo26n-seg.pt",  
    classes=[0], 
)


while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    results = isegment(im0)
    video_writer.write(results.plot_im) 

cap.release()
video_writer.release()
cv2.destroyAllWindows()  

In [ ]:
# Q3 D
# https://docs.ultralytics.com/guides/object-counting

import cv2
from ultralytics import solutions, YOLO

video = "./q3/video/people.mp4"
cap = cv2.VideoCapture(video)

if not cap.isOpened():
    print("Error: Could not open video files")
    exit()

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("q3/results/object_counting_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Entire video frame is the region of interest for counting, so we define the four corners of the frame as a polygon
# but reduced by 200px all around to not count objects that are too close to the edge of the frame
region_points = [(0+200, 0+200), (w-200, 0+200), (w-200, h-200), (0+200, h-200)]

counter = solutions.ObjectCounter(
    show=False, 
    region=region_points, 
    model="yolo26n.pt",
    classes=[0],  
)

while True:
    success, im0 = cap.read()

    if not success:
        break

    results = counter(im0)
    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()



# Q4. Classification                                                        10  marks
Given the data-set fault-classification-class. Perform the following
(a) Develop ML model to determine the state of health of asset (in this case motor).
(b) Show, train and val loss, confusion matrix and classification report
(c) Consider the one in the class as baseline and compare your model with it. Summarize in terms of
model parameter, inference time, accuracy, precision, recall, F1 score

# Q5. Classification                                                        20  
Download CFAR-10 dataset (https://www.tensorflow.org/datasets/catalog/cifar10). Perform the classification task on this dataset 
(a) Use model fusion (use two or models to extract feature and combine them), 
(b) Use transfer learning
(c) Use finetuning
(d) Compare different methods (a) - (c) in terms of typical KPIs often used for classification

# Q6. Autoencoder                                                           10 marks
Given the data-set fault-classification-class, 
Use Auto encoder to compress the data to latent space and reconstruct it back. Show the following for random 10 iamges from each class
(a) Original image
(b) Latent vector
(c) Reconstructed back image
(d) individual reconstruction error and mean loss for each class.
